# CS224N · Lecture 13 — Speech and Brain–Computer Interfaces

**Slides:** [cs224n-spr2024-lecture13-speech-bci.pdf](https://web.stanford.edu/class/cs224n/slides/cs224n-spr2024-lecture13-speech-bci.pdf)

> ⚠️ **About this notebook.** The slide file for this lecture (62 MB) could not be retrieved when this notebook was written, so unlike the other notebooks in this series, **it is not a walkthrough of the slides.** It covers the technical core of the topic as published in the research literature: how speech recognition works (features, the **CTC** loss, language-model decoding, word error rate), and how the same machinery powers **speech brain–computer interfaces (BCIs)** that decode attempted speech from neural activity. Facts about specific BCI studies are taken from the published papers cited in each section. If you have the slides, use them to see which parts the lecture emphasized.

| § | Topic | You will implement / measure |
|---|---|---|
| 1 | From sound to features | A log-mel spectrogram from scratch |
| 2 | The alignment problem and **CTC** | CTC loss with forward–backward, gradient-checked; greedy decoding |
| 3 | Evaluating recognizers: WER / PER | Edit distance |
| 4 | Language models in decoding | Rescoring with a word bigram LM |
| 5 | **Speech BCIs**: the problem and the published systems | — |
| 6 | **A simulated speech BCI** | An LSTM + CTC decoder on synthetic neural data; LM decoding; **day-to-day drift** and recalibration |
| 7 | 🔎 Modern speech models (wav2vec 2.0, Whisper) and open problems | — |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random, math, time, collections

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))
def log_softmax(x, axis=-1):
    m = x.max(axis=axis, keepdims=True); return x - m - np.log(np.exp(x - m).sum(axis=axis, keepdims=True))

---
## 1. From sound to features

Speech is a pressure wave sampled at, e.g., 16,000 samples per second. Raw samples are a poor input representation (one second = 16,000 numbers, mostly encoding fine phase detail that doesn't matter for *what* was said). The classic front end:
1. Cut the signal into short overlapping **frames** (25 ms windows every 10 ms); speech is roughly stationary over such short spans.
2. Take each frame's **Fourier transform** (the short-time Fourier transform, STFT) → the energy at each frequency.
3. Pool frequencies with a **mel filterbank**: triangular filters spaced evenly on the **mel scale** $m = 2595\log_{10}(1 + f/700)$, which mimics human pitch perception (finer resolution at low frequencies).
4. Take the **log** (loudness perception is roughly logarithmic, and it compresses the dynamic range).

The result, a **log-mel spectrogram** (~80 numbers per 10 ms), is the input to most modern recognizers, including Whisper.

In [ ]:
SR = 16000
t = np.arange(int(0.8 * SR)) / SR
signal = (np.sin(2 * np.pi * (300 + 1500 * t) * t) * (t < 0.4)                     # a rising chirp (like a formant transition)
          + 0.6 * np.sin(2 * np.pi * 700 * t) * (t >= 0.4) + 0.4 * np.sin(2 * np.pi * 2200 * t) * (t >= 0.4)   # two steady "formants"
          + 0.05 * rng.normal(size=len(t)))

def stft_power(x, win=400, hop=160):                       # 25 ms windows, 10 ms hop at 16 kHz
    w = np.hanning(win)
    frames = np.stack([x[i:i + win] * w for i in range(0, len(x) - win, hop)])
    return np.abs(np.fft.rfft(frames, n=512)) ** 2          # (frames, 257 frequency bins)

def mel_filterbank(n_mels=40, n_fft=512, sr=SR):
    mel = lambda f: 2595 * np.log10(1 + f / 700); inv = lambda m: 700 * (10 ** (m / 2595) - 1)
    pts = inv(np.linspace(mel(0), mel(sr / 2), n_mels + 2))                         # evenly spaced on the mel scale
    bins = np.floor((n_fft + 1) * pts / sr).astype(int)
    fb = np.zeros((n_mels, n_fft // 2 + 1))
    for m in range(1, n_mels + 1):
        l, c, r_ = bins[m - 1], bins[m], bins[m + 1]
        fb[m - 1, l:c] = (np.arange(l, c) - l) / max(c - l, 1); fb[m - 1, c:r_] = (r_ - np.arange(c, r_)) / max(r_ - c, 1)
    return fb

P = stft_power(signal); logmel = np.log(P @ mel_filterbank().T + 1e-10)
print(f"0.8 s of audio = {len(signal):,} samples -> log-mel spectrogram of shape {logmel.shape} (frames x mel bands)")
plt.figure(figsize=(8, 3)); plt.imshow(logmel.T, origin="lower", aspect="auto", cmap="magma")
plt.xlabel("frame (10 ms)"); plt.ylabel("mel band"); plt.title("Log-mel spectrogram: a chirp, then two steady tones"); plt.colorbar(); plt.show()

---
## 2. The alignment problem, and CTC

An utterance gives ~100 feature frames per second, but only ~10–15 **phonemes** (speech sounds) per second, and we don't know which frames belong to which phoneme: **the input and output sequences have different lengths and an unknown alignment.** (NMT handled this with an encoder–decoder + attention, L7. Speech recognition's classic answer is CTC.)

**Connectionist Temporal Classification** (Graves, Fernández, Gomez & Schmidhuber, ICML 2006; L6 noted it was invented in the same line of work that made LSTMs famous):
* At every frame, the network outputs a distribution over the labels **plus a special blank** symbol "‑".
* A frame-level path (e.g., `‑ k k ‑ æ æ ‑ t`) maps to a label sequence by **collapsing repeats, then removing blanks**: `k æ t`. (A blank between two identical labels keeps them separate: `l ‑ l` → `l l`.)
* The probability of a label sequence $y$ is the **sum over all paths** that collapse to it: $P(y\mid x) = \sum_{\pi\in\mathcal{B}^{-1}(y)}\prod_t p_t(\pi_t)$, assuming frames are conditionally independent given the input.
* The loss is $-\log P(y\mid x)$. The sum over exponentially many paths is computed by **dynamic programming** (the forward–backward algorithm, as for HMMs) over the extended label sequence $y' = (‑, y_1, ‑, y_2, \dots, ‑)$ of length $2|y|+1$.

**Forward recursion** (in log space for stability), with $\alpha_t(s)$ = log-probability of all paths that end at position $s$ of $y'$ at frame $t$:
$$\alpha_t(s) = \log p_t(y'_s) + \text{logsumexp}\big(\alpha_{t-1}(s),\ \alpha_{t-1}(s-1),\ \alpha_{t-1}(s-2)\ [\text{if } y'_s\neq ‑ \text{ and } y'_s\neq y'_{s-2}]\big)$$
$\log P(y\mid x) = \text{logsumexp}(\alpha_T(S), \alpha_T(S-1))$. A matching **backward** recursion gives $\beta_t(s)$; their product gives the posterior probability $\gamma_t(s)$ that frame $t$ is at position $s$. The gradient with respect to the **logits** is then simply
$$\frac{\partial(-\log P)}{\partial z_t(k)} = p_t(k) - \sum_{s:\,y'_s = k}\gamma_t(s)$$
**predicted distribution minus the "soft target"** given by the alignment posterior: the same "prediction minus target" pattern as softmax + cross-entropy (L3), with the target computed by dynamic programming.

In [ ]:
BLANK = 0

def lse(*xs):
    m = max(xs)
    return m if m == -np.inf else m + math.log(sum(math.exp(x - m) for x in xs))

def ctc_loss_grad(logits, labels):
    # logits: (T, C) for ONE sequence; labels: list of ints (no blanks). Returns loss, dloss/dlogits.
    T, C = logits.shape; lp = log_softmax(logits)
    ext = [BLANK]
    for l in labels:
        ext += [l, BLANK]
    S = len(ext); NEG = -np.inf
    alpha = np.full((T, S), NEG); beta = np.full((T, S), NEG)
    alpha[0, 0] = lp[0, BLANK]
    if S > 1:
        alpha[0, 1] = lp[0, ext[1]]
    for t in range(1, T):
        for s in range(S):
            terms = [alpha[t - 1, s]] + ([alpha[t - 1, s - 1]] if s >= 1 else [])
            if s >= 2 and ext[s] != BLANK and ext[s] != ext[s - 2]:
                terms.append(alpha[t - 1, s - 2])
            alpha[t, s] = lse(*terms) + lp[t, ext[s]]
    beta[T - 1, S - 1] = lp[T - 1, ext[S - 1]]
    if S > 1:
        beta[T - 1, S - 2] = lp[T - 1, ext[S - 2]]
    for t in range(T - 2, -1, -1):
        for s in range(S):
            terms = [beta[t + 1, s]] + ([beta[t + 1, s + 1]] if s + 1 < S else [])
            if s + 2 < S and ext[s] != BLANK and ext[s] != ext[s + 2]:
                terms.append(beta[t + 1, s + 2])
            beta[t, s] = lse(*terms) + lp[t, ext[s]]
    logP = lse(alpha[T - 1, S - 1], alpha[T - 1, S - 2]) if S > 1 else alpha[T - 1, 0]
    gamma = np.exp(alpha + beta - lp[:, ext] - logP)            # posterior of being at extended position s at frame t
    grad = np.exp(lp)
    for s, k in enumerate(ext):
        grad[:, k] -= gamma[:, s]
    return -logP, grad

def brute_force_ctc(logits, labels):
    # Sum over ALL frame paths (tiny cases only) to check the dynamic program.
    import itertools
    T, C = logits.shape; p = np.exp(log_softmax(logits)); total = 0.0
    for path in itertools.product(range(C), repeat=T):
        collapsed = [k for i, k in enumerate(path) if (i == 0 or k != path[i - 1])]
        if [k for k in collapsed if k != BLANK] == labels:
            total += np.prod(p[np.arange(T), path])
    return -math.log(total)

r = np.random.default_rng(1)
z = r.normal(size=(5, 3)); lab = [1, 2]
loss, g = ctc_loss_grad(z, lab)
print(f"CTC loss: dynamic programming {loss:.6f}   brute force over all {3 ** 5} paths {brute_force_ctc(z, lab):.6f}")
lab2 = [1, 1]                                                     # repeated label: needs a blank in between
print(f"repeated labels [1, 1]: DP {ctc_loss_grad(z, lab2)[0]:.6f}   brute force {brute_force_ctc(z, lab2):.6f}")
num = np.zeros_like(z)
for idx in np.ndindex(z.shape):
    old = z[idx]; z[idx] = old + 1e-6; lp_ = ctc_loss_grad(z, lab)[0]; z[idx] = old - 1e-6; lm_ = ctc_loss_grad(z, lab)[0]; z[idx] = old
    num[idx] = (lp_ - lm_) / 2e-6
print(f"gradient check: max |analytic - numerical| = {np.abs(num - g).max():.1e}")

def greedy_ctc_decode(logits):
    best = logits.argmax(-1)
    return [int(k) for i, k in enumerate(best) if k != BLANK and (i == 0 or k != best[i - 1])]

🔎 **CTC's assumption and its alternatives.** CTC assumes each frame's output is conditionally independent of the others given the input, so a CTC model has no internal language model; it's usually combined with an external LM (§4). The **RNN-Transducer** (Graves 2012) adds a prediction network over previous labels (used in on-device recognizers), and **attention encoder–decoders** (e.g., Whisper, §7) generate the transcript autoregressively like NMT.

---
## 3. Evaluating recognizers: word error rate

**Word error rate** = the minimum number of **substitutions + deletions + insertions** needed to turn the hypothesis into the reference, divided by the number of reference words (the Levenshtein edit distance, computed by dynamic programming). **Phoneme error rate (PER)** is the same over phonemes. WER can exceed 100% (many insertions). Note it treats every word error equally: "a" vs. "the" costs as much as a wrong name.

In [ ]:
def edit_distance(ref, hyp):
    d = np.arange(len(hyp) + 1)
    for i in range(1, len(ref) + 1):
        prev, d[0] = d.copy(), i
        for j in range(1, len(hyp) + 1):
            d[j] = min(prev[j] + 1, d[j - 1] + 1, prev[j - 1] + (ref[i - 1] != hyp[j - 1]))   # deletion, insertion, substitution
    return int(d[-1])

def error_rate(refs, hyps):
    return sum(edit_distance(r_, h_) for r_, h_ in zip(refs, hyps)) / sum(len(r_) for r_ in refs)

ref = "i would like a cup of tea".split()
for hyp in ["i would like a cup of tea", "i would like cup of tea", "i would like a cap of the tea", "eye wood like a cup of tea"]:
    print(f"WER {error_rate([ref], [hyp.split()]):5.1%}   {hyp}")

---
## 4. Language models in decoding

An acoustic model (or neural decoder) alone confuses similar-sounding words ("eye wood" for "I would"). Decoding combines it with a **language model**, as SMT did (L6: fidelity × fluency):
$$\hat W = \arg\max_W\ \log P_{\text{acoustic}}(W\mid x) + \lambda\log P_{\text{LM}}(W) + \mu|W|$$
($\lambda$ weighs the LM, $\mu$ is a word-insertion bonus). In practice this runs inside a **beam search** over a lexicon-constrained graph (often a weighted finite-state transducer), with an n-gram LM; a large neural LM can then **rescore** the top hypotheses. We'll use a simpler version in §6: map each decoded phoneme chunk to candidate lexicon words, and pick the word sequence that best trades off phoneme edit distance against a word bigram LM.

---
## 5. Speech brain–computer interfaces

**The problem.** People with paralysis from ALS (amyotrophic lateral sclerosis) or brainstem stroke can lose the ability to speak, while the brain areas that plan and control speech remain active. A **speech neuroprosthesis** records neural activity while the person *attempts* to speak, and decodes it into text (or synthesized speech).

**How it's built (published systems):**
* **Recording:** **intracortical microelectrode arrays** (e.g., Utah arrays, 64–96 electrodes each) implanted in speech motor cortex, or **ECoG** grids on the brain surface.
* **Features:** for each electrode, in short time bins (e.g., 20 ms): **threshold crossings** (approximate spike counts) and **spike-band power**.
* **Decoder:** a **recurrent neural network** maps the neural feature sequence to **phoneme probabilities**, trained with **CTC** because the timing of each phoneme in the attempted speech is unknown (the participant can't produce audio to align to).
* **Language model:** decode phoneme probabilities into words with an n-gram LM and beam search, optionally rescored by a large LM.
* **Recalibration:** neural recordings **drift** from day to day (electrodes move slightly, neurons change), so systems use **day-specific input layers** and recalibrate with a little new data each session.

**Results reported in the literature:**
* **Willett et al. 2023, *Nature*** (Stanford; a participant with ALS; four arrays in ventral premotor cortex (area 6v) and Broca's area (area 44)): decoding attempted speech at **62 words per minute**, with **9.1% WER on a 50-word vocabulary** and **23.8% WER on a 125,000-word vocabulary**. Most of the decodable information came from area 6v.
* **Metzger et al. 2023, *Nature*** (UCSF; ECoG; a participant with brainstem stroke): text decoding at ~78 words per minute, plus speech synthesis and an animated facial avatar.
* **Card et al. 2024, *NEJM*** (UC Davis; a participant with ALS; four arrays): **97.5% accuracy** (≈2.5% WER) with a **125,000-word** vocabulary, usable for conversation from early sessions, with ongoing recalibration.

The ML is exactly this course's toolkit (RNNs, CTC, n-gram and neural LMs, beam search, LLM rescoring), applied under severe constraints: small datasets (hours of attempted speech, not thousands), non-stationary inputs, real-time latency, and very high stakes for the person using it.

---
## 6. A simulated speech BCI

Real neural data isn't available here, so we **simulate** recordings from a simple, commonly assumed model: each recording channel's activity is **tuned** to the phoneme being attempted (a phoneme-specific mean pattern across channels), plus noise. Sentences are generated from a word bigram model over a 30-word lexicon; each word is a sequence of phonemes, and words are separated by a silence/word-boundary phoneme "|".

In [ ]:
r = np.random.default_rng(0)
N_PH = 16                                                         # phoneme inventory (0 = CTC blank, 1 = word boundary "|")
PH = ["-", "|"] + [f"p{i}" for i in range(2, N_PH)]
lexicon = {}
while len(lexicon) < 30:                                          # 30 words, each 2-4 phonemes
    pron = tuple(int(x) for x in r.integers(2, N_PH, int(r.integers(2, 5))))
    lexicon.setdefault(pron, f"w{len(lexicon)}")
words = list(lexicon.values()); pron_of = {w: p for p, w in lexicon.items()}
bigram = r.dirichlet(np.full(len(words), 0.15), len(words))       # a peaky word bigram LM: P(next | prev)
start = r.dirichlet(np.full(len(words), 0.5))

def sample_sentence(r_):
    n = int(r_.integers(3, 6)); w = [int(r_.choice(len(words), p=start))]
    for _ in range(n - 1):
        w.append(int(r_.choice(len(words), p=bigram[w[-1]])))
    return [words[i] for i in w]

N_CH = 64
tuning = r.normal(0, 1, (N_PH, N_CH))                             # each phoneme's mean activity pattern across channels

def simulate_neural(sentence, r_, transform=None):
    phones = []
    for i, w in enumerate(sentence):
        phones += list(pron_of[w]) + ([1] if i < len(sentence) - 1 else [])
    frames = []
    for ph in phones:
        for _ in range(int(r_.integers(3, 7))):                    # each phoneme lasts 3-6 bins (60-120 ms)
            frames.append(tuning[ph] + r_.normal(0, 1.6, N_CH))    # noisy, phoneme-tuned activity
    X = np.array(frames)
    if transform is not None:                                     # day-to-day drift
        X = X @ transform[0] + transform[1]
    return X, phones

ex_sent = sample_sentence(r); X_ex, ph_ex = simulate_neural(ex_sent, r)
print("sentence:", " ".join(ex_sent)); print("phonemes:", " ".join(PH[p] for p in ph_ex))
print(f"neural features: {X_ex.shape} (time bins x channels)")

**The decoder:** a linear input layer (the "day-specific" layer), then an LSTM (L6's implementation, with masking for variable lengths), then a linear layer to logits over {blank, "|", phonemes}, trained with the CTC loss above. The CTC gradient flows into the LSTM through its per-timestep hidden-state gradients.

In [ ]:
def lstm_forward(W, b, X, mask, h0, c0):
    B, T, d = X.shape; H = h0.shape[1]; h, c = h0, c0; Hs = np.zeros((B, T, H)); cache = []
    for t in range(T):
        xh = np.concatenate([X[:, t], h], 1); a = xh @ W.T + b
        f, i, o = sigmoid(a[:, :H]), sigmoid(a[:, H:2 * H]), sigmoid(a[:, 2 * H:3 * H]); g = np.tanh(a[:, 3 * H:])
        cn = f * c + i * g; tc = np.tanh(cn); hn = o * tc; m = mask[:, t:t + 1]
        cache.append((xh, f, i, o, g, c, tc, m)); h = m * hn + (1 - m) * h; c = m * cn + (1 - m) * c; Hs[:, t] = h
    return Hs, cache

def lstm_backward(W, dHs, cache, d):
    B, T, H = dHs.shape; dX = np.zeros((B, T, d)); gW = np.zeros_like(W); gb = np.zeros(W.shape[0])
    dh, dc = np.zeros((B, H)), np.zeros((B, H))
    for t in reversed(range(T)):
        xh, f, i, o, g, c_prev, tc, m = cache[t]; dh = dh + dHs[:, t]
        dhn, dcin = m * dh, m * dc; dh_prev, dc_prev = (1 - m) * dh, (1 - m) * dc
        do = dhn * tc; dct = dhn * o * (1 - tc ** 2) + dcin
        df, di, dg = dct * c_prev, dct * g, dct * i; dc_prev = dc_prev + dct * f
        da = np.concatenate([df * f * (1 - f), di * i * (1 - i), do * o * (1 - o), dg * (1 - g ** 2)], 1)
        gW += da.T @ xh; gb += da.sum(0); dxh = da @ W; dX[:, t] = dxh[:, :d]; dh_prev = dh_prev + dxh[:, d:]
        dh, dc = dh_prev, dc_prev
    return dX, gW, gb

D_IN, H_L = 32, 64
def init_decoder(seed=0):
    r_ = np.random.default_rng(seed)
    p = {"Win": r_.normal(0, 1 / np.sqrt(N_CH), (N_CH, D_IN)), "bin": np.zeros(D_IN),
         "W": r_.normal(0, 1 / np.sqrt(D_IN + H_L), (4 * H_L, D_IN + H_L)), "b": np.zeros(4 * H_L),
         "U": r_.normal(0, 1 / np.sqrt(H_L), (H_L, N_PH)), "c": np.zeros(N_PH)}
    p["b"][:H_L] = 1.0
    return p

def batch(examples):
    T = max(len(X) for X, _ in examples); B = len(examples)
    Xb = np.zeros((B, T, N_CH)); M = np.zeros((B, T))
    for i, (X, _) in enumerate(examples):
        Xb[i, :len(X)] = X; M[i, :len(X)] = 1
    return Xb, M

def decoder_loss_grads(p, examples, train_keys):
    Xb, M = batch(examples); B, T, _ = Xb.shape
    Z = Xb @ p["Win"] + p["bin"]                                   # day-specific input layer
    Hs, cache = lstm_forward(p["W"], p["b"], Z, M, np.zeros((B, H_L)), np.zeros((B, H_L)))
    logits = Hs @ p["U"] + p["c"]
    total, dlogits = 0.0, np.zeros_like(logits)
    for i, (X, phones) in enumerate(examples):
        l, g = ctc_loss_grad(logits[i, :len(X)], phones)
        total += l / len(phones); dlogits[i, :len(X)] = g / len(phones) / B
    g = {"U": Hs.reshape(-1, H_L).T @ dlogits.reshape(-1, N_PH), "c": dlogits.sum((0, 1))}
    dZ, g["W"], g["b"] = lstm_backward(p["W"], dlogits @ p["U"].T, cache, D_IN)
    g["Win"] = Xb.reshape(-1, N_CH).T @ dZ.reshape(-1, D_IN); g["bin"] = dZ.sum((0, 1))
    return total / B, {k: g[k] for k in train_keys}, logits

def adam_train(p, data, keys, steps, lr=3e-3, batch_size=16, seed=0, log_every=None):
    r_ = np.random.default_rng(seed); m = {k: np.zeros_like(p[k]) for k in keys}; v = {k: np.zeros_like(p[k]) for k in keys}
    for t in range(1, steps + 1):
        loss, g, _ = decoder_loss_grads(p, [data[i] for i in r_.integers(0, len(data), batch_size)], keys)
        norm = math.sqrt(sum((x ** 2).sum() for x in g.values()))
        for k in keys:
            gk = g[k] * min(1.0, 5.0 / (norm + 1e-12))
            m[k] = 0.9 * m[k] + 0.1 * gk; v[k] = 0.999 * v[k] + 0.001 * gk ** 2
            p[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)
        if log_every and t % log_every == 0:
            print(f"  step {t}: CTC loss per phoneme {loss:.3f}")
    return p

r_data = np.random.default_rng(1)
day1_sents = [sample_sentence(r_data) for _ in range(500)]
day1 = [simulate_neural(s, r_data) for s in day1_sents]
test_sents = [sample_sentence(r_data) for _ in range(100)]
test_day1 = [simulate_neural(s, r_data) for s in test_sents]

t0 = time.time()
p_dec = adam_train(init_decoder(), day1, ["Win", "bin", "W", "b", "U", "c"], steps=400, log_every=100)
print(f"trained in {time.time() - t0:.0f}s")

In [ ]:
def decode_phonemes(p, examples):
    outs = []
    for i in range(0, len(examples), 25):
        chunk = examples[i:i + 25]; _, _, logits = decoder_loss_grads(p, chunk, [])
        outs += [greedy_ctc_decode(logits[j, :len(X)]) for j, (X, _) in enumerate(chunk)]
    return outs

def phonemes_to_words(phones, use_lm, lm_weight=1.5):
    # Split on word boundaries, then pick words: nearest pronunciation, optionally rescored with the bigram LM (Viterbi).
    chunks = [[]]
    for ph in phones:
        if ph == 1: chunks.append([])
        else: chunks[-1].append(ph)
    chunks = [c for c in chunks if c]
    if not chunks:
        return []
    cost = np.array([[edit_distance(list(pron_of[w]), c) for w in words] for c in chunks], float)   # acoustic cost
    if not use_lm:
        return [words[j] for j in cost.argmin(1)]
    logb, logs = np.log(bigram + 1e-12), np.log(start + 1e-12)
    score = -cost[0] + lm_weight * logs; back = []
    for k in range(1, len(chunks)):                                  # Viterbi over word sequences
        cand = score[:, None] + lm_weight * logb                       # (prev, next)
        back.append(cand.argmax(0)); score = cand.max(0) - cost[k]
    best = [int(score.argmax())]
    for bp in reversed(back):
        best.append(int(bp[best[-1]]))
    return [words[j] for j in reversed(best)]

def evaluate(p, examples, sents, lm_weight):
    hyp_ph = decode_phonemes(p, examples)
    per = error_rate([ph for _, ph in examples], hyp_ph)
    wer_no = error_rate(sents, [phonemes_to_words(h, False) for h in hyp_ph])
    wer_lm = error_rate(sents, [phonemes_to_words(h, True, lm_weight) for h in hyp_ph])
    return per, wer_no, wer_lm

# The LM weight must be TUNED on held-out data: too large and the LM overrides correct acoustic evidence.
dev_sents = [sample_sentence(r_data) for _ in range(100)]
dev_day1 = [simulate_neural(s, r_data) for s in dev_sents]
dev_hyp = decode_phonemes(p_dec, dev_day1)
sweep = {lam: error_rate(dev_sents, [phonemes_to_words(h, True, lam) for h in dev_hyp]) for lam in [0.0, 0.05, 0.1, 0.2, 0.4, 0.8, 1.5]}
print("dev-set WER by LM weight:", {k: f"{v:.1%}" for k, v in sweep.items()})
LM_WEIGHT = min(sweep, key=sweep.get)
per, wer_no, wer_lm = evaluate(p_dec, test_day1, test_sents, LM_WEIGHT)
print(f"day 1 test: PER {per:.1%}   WER without LM {wer_no:.1%}   WER with bigram LM (weight {LM_WEIGHT}) {wer_lm:.1%}")

### 6.1 Day-to-day drift and recalibration
On a new day, the recorded signals change: each channel's sensitivity and baseline shift, and the mapping from neurons to electrodes changes slightly. We simulate "day 2" with a random linear transformation of the channels (mixing, rescaling, offsets). The same decoder, unchanged, then sees inputs it was never trained on.

The fix used in published speech BCIs: keep the network, and **retrain only a day-specific input layer** on a small amount of new data. Here: the first linear layer (`Win`, `bin`), with only 40 new sentences, while the LSTM and output layer stay frozen.

In [ ]:
r_d2 = np.random.default_rng(2)
drift = (np.eye(N_CH) + 0.6 * r_d2.normal(0, 1 / np.sqrt(N_CH), (N_CH, N_CH))) * r_d2.uniform(0.6, 1.4, N_CH), r_d2.normal(0, 0.8, N_CH)
test_day2 = [simulate_neural(s, r_d2, drift) for s in test_sents]
calib_day2 = [simulate_neural(sample_sentence(r_d2), r_d2, drift) for _ in range(40)]

per_u, wno_u, wlm_u = evaluate(p_dec, test_day2, test_sents, LM_WEIGHT)
print(f"day 2, decoder unchanged      : PER {per_u:.1%}   WER without LM {wno_u:.1%}   WER with LM {wlm_u:.1%}")
p_recal = {k: v.copy() for k, v in p_dec.items()}
t0 = time.time()
p_recal = adam_train(p_recal, calib_day2, ["Win", "bin"], steps=150, lr=3e-3, seed=5)
per2, wno2, wer2 = evaluate(p_recal, test_day2, test_sents, LM_WEIGHT)
print(f"day 2, input layer recalibrated on 40 sentences: PER {per2:.1%}   WER without LM {wno2:.1%}   WER with LM {wer2:.1%}   ({time.time() - t0:.0f}s)")

What the simulation shows, in miniature:
* **CTC learns phoneme sequences without any timing labels**: we never told the decoder which bins belong to which phoneme.
* **The language model helps only when its weight is tuned.** With a large weight, the bigram prior overrides correct acoustic evidence whenever the true word was an unlikely continuation, and WER gets *worse* (see the dev-set sweep). Tuned on held-out data, the LM resolves confusable words using context, as in speech recognition and the BCI papers' n-gram/LLM decoding stages. How much it helps depends on how noisy the phonemes are: the noisier the decoder, the more the LM has to contribute.
* **Drift more than doubles the error rate of an unchanged decoder**, and **recalibrating only the input layer** with a few dozen sentences recovers most of the performance, because the "phoneme → sequence" knowledge in the LSTM is unchanged; only the mapping from electrodes to the decoder's internal representation moved. (The published systems rely on this idea; cross-participant pretraining of decoders, with per-participant and per-day input layers, is an active research direction.)

All numbers here are from a simplified simulation: real neural data is far noisier, non-linear, and non-stationary in more complex ways.

---
## 7. 🔎 Modern speech recognition, and open problems

* **Self-supervised speech pretraining** (L9's idea, for audio): **wav2vec 2.0** (Baevski et al. 2020) learns representations from raw audio by masking spans and solving a contrastive task (L9 §4.3's InfoNCE) against quantized targets; finetuned with CTC, it works with very little labeled speech. HuBERT is similar, with offline-clustered targets.
* **Whisper** (Radford et al. 2022): a Transformer **encoder–decoder** (L8) on log-mel spectrograms (§1), trained on ~680,000 hours of weakly supervised (web) audio–transcript pairs, multitask (transcription, translation, language ID, timestamps) via special tokens in the decoder prompt. Robust across accents and noise, without dataset-specific finetuning.
* **Speech-native LLMs:** models that take and produce audio tokens directly (audio tokenizers/codecs + a language model), enabling spoken dialogue without a separate recognizer and synthesizer.
* **Open problems:** low-resource languages and dialects; fairness (WER differs across accents and demographic groups); robustness to noise and far-field audio; and for BCIs, decoding with less calibration data, long-term stability, and privacy of neural data.

In [ ]:
# [Optional] needs: pip install transformers torch librosa  (downloads ~150MB). Whisper on any 16 kHz audio array.
try:
    from transformers import pipeline
    asr = pipeline("automatic-speech-recognition", model="openai/whisper-tiny")
    import numpy as _np
    tone = 0.1 * _np.sin(2 * _np.pi * 440 * _np.arange(16000) / 16000).astype(_np.float32)   # replace with real speech
    print(asr({"raw": tone, "sampling_rate": 16000}))
except Exception as e:
    print("Skipping Whisper demo:", e)

---
## 8. Summary

| Topic | Takeaway |
|---|---|
| Features | Frames → FFT → mel filterbank → log: the log-mel spectrogram |
| Alignment | Input frames ≫ output labels, timing unknown |
| CTC | Blank + collapse; sum over paths by forward–backward; gradient = softmax − alignment posterior |
| WER / PER | Edit distance (subs + dels + ins) / reference length |
| LM decoding | $\log P_{ac} + \lambda\log P_{LM}$; beam search / Viterbi; LLM rescoring |
| Speech BCIs | Intracortical arrays → binned spike features → RNN + CTC → phonemes → LM → text; published: 62 wpm / 23.8% WER (2023), 97.5% accuracy (2024), 125K vocabulary |
| Drift | Day-specific input layers, recalibrated with little data |
| 🔎 Modern ASR | wav2vec 2.0 (self-supervised + CTC), Whisper (weakly supervised encoder–decoder) |

### Exercises
1. Implement **CTC prefix beam search** (track blank-ending and non-blank-ending probabilities per prefix) and compare with greedy decoding on the simulated BCI.
2. Fold the LM into decoding at the **phoneme level** (shallow fusion inside the beam) instead of rescoring after segmentation.
3. Increase the noise level in `simulate_neural` and plot PER vs. noise. At what point does the LM stop compensating?
4. Recalibrate with 5, 10, 20, 40, 80 sentences and plot the day-2 WER. How little data is enough?
5. Make the drift **non-linear** (e.g., some channels die: set them to zero). Does an affine input layer still recover?

### References
* Graves, Fernández, Gomez & Schmidhuber 2006, *Connectionist Temporal Classification*; Graves 2012 (RNN-Transducer); Hannun 2017, *Sequence Modeling with CTC* (Distill).
* Jurafsky & Martin, *Speech and Language Processing* (3rd ed.), Ch. 16 "Automatic Speech Recognition and Text-to-Speech".
* Willett et al. 2023, *A high-performance speech neuroprosthesis* (Nature); Metzger et al. 2023, *A high-performance neuroprosthesis for speech decoding and avatar control* (Nature); Card et al. 2024, *An accurate and rapidly calibrating speech neuroprosthesis* (NEJM).
* Baevski et al. 2020 (wav2vec 2.0); Hsu et al. 2021 (HuBERT); Radford et al. 2022 (Whisper).